# 1. Setup & Environment
- 공간 통계(PySAL, ESDA) 및 공간 데이터프레임(GeoPandas) 라이브러리 로드
- 부동소수점 출력 포맷 및 판다스 디스플레이 환경 설정

In [1]:
# 1. 라이브러리 로드 및 환경 설정
from pathlib import Path
import warnings

import esda
import geopandas as gpd
import libpysal
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: "%.4f" % x)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

# 2. Configuration & Comparative Model Definitions
- 서울시 집계구 경계 및 2개 모형(2SFCA, Gravity) 점수 디렉터리 설정
- 랩미팅 확정 파라미터 정의: 평일 낮(week_낮_normal) 단일 시나리오, 4개년(2021~2024), KNN(k=30)
- 주의: 모형 간 광역 패턴 비교 목적이므로 우선설치 진단(k=8)과 구분하여 k=30 적용

In [2]:
# 2. 경로 및 분석 파라미터 설정
BASE_DIR = Path("/mnt/cowork/EV")
BOUNDARY_FP = BASE_DIR / "input/raw/집계구_2016/집계구.shp"

DIR_2SFCA = BASE_DIR / "output/g2sfca_sfast_final_gaussian"
DIR_GRAVITY = BASE_DIR / "output/gravity_model_gaussian"
DIR_OUTPUT = BASE_DIR / "output"
DIR_OUTPUT.mkdir(parents=True, exist_ok=True)

# 모델링 파라미터
YEARS = [2021, 2022, 2023, 2024]
SCENARIO_TAG = "week_낮_normal"
K_NEIGHBORS = 30  # 광역 공간 패턴 비교용 KNN
P_THRESHOLD = 0.05

print(f">> 비교 모형: Gaussian 2SFCA vs Gravity Model (총 2개 모형)")
print(f">> 분석 시나리오: {SCENARIO_TAG} ({YEARS}개년, 총 {len(YEARS)*2}개 조합)")
print(f">> 공간가중 파라미터: KNN (k={K_NEIGHBORS}), 유의수준: p < {P_THRESHOLD}")

>> 비교 모형: Gaussian 2SFCA vs Gravity Model (총 2개 모형)
>> 분석 시나리오: week_낮_normal ([2021, 2022, 2023, 2024]개년, 총 8개 조합)
>> 공간가중 파라미터: KNN (k=30), 유의수준: p < 0.05


# 3. Spatial Weights & Local Gi* Analytics Engine
- 집계구 중심점 기준 KNN(k=30) 공간가중행렬 생성 함수
- 이진 가중(transform="B") 기반 Local Getis-Ord Gi* 통계량 산출 및 Hot/Cold/Not Sig 분류 함수

In [3]:
# 3. 공간가중행렬 및 Gi* 연산 엔진
def build_knn_weights(gdf: gpd.GeoDataFrame, k: int = K_NEIGHBORS) -> libpysal.weights.KNN:
    """집계구 Centroid 기준 KNN 공간가중행렬 생성"""
    centroids = np.array([[geom.centroid.x, geom.centroid.y] for geom in gdf.geometry])
    return libpysal.weights.KNN.from_array(centroids, k=k)


def compute_gi_star(y_scores: np.ndarray, w_spatial: libpysal.weights.W, p_th: float = P_THRESHOLD) -> np.ndarray:
    """Local Gi* 연산 및 Hot/Cold Spot 3분할 범주형 분류"""
    # Binary 공간가중 변환 기반 Local Gi* 산출 (다중검정 미보정 표준 유지)
    lg = esda.getisord.G_Local(y_scores, w_spatial, transform="B")
    
    coded = np.where(
        (lg.Zs < 0) & (lg.p_norm < p_th), "Cold Spot",
        np.where((lg.Zs > 0) & (lg.p_norm < p_th), "Hot Spot", "Not Sig")
    )
    return coded

# 4. Boundary Loader & Model Score Path Resolver
- 서울시 집계구(EPSG:5179) 폴리곤 로드 및 전처리
- 2SFCA 및 Gravity 산출물 파일 경로 매퍼 (_mw 우선, 구버전 호환)

In [4]:
# 4. 경계 로더 및 점수 파일 매퍼
def load_seoul_boundary(boundary_path: Path) -> gpd.GeoDataFrame:
    """서울시 집계구(코드 11 시작) 경계 로드 및 좌표계 설정"""
    gdf = gpd.read_file(boundary_path)
    gdf = gdf.set_crs(epsg=5179, allow_override=True)
    gdf["TOT_REG_CD"] = gdf["TOT_REG_CD"].astype(str)
    gdf = gdf[gdf["TOT_REG_CD"].str.startswith("11")].copy().reset_index(drop=True)
    return gdf


def resolve_model_score_file(model_name: str, year: int) -> Path:
    """모형별 접근성 점수 CSV 경로 탐색 (_mw 우선)"""
    if model_name == "2SFCA":
        fp_mw = DIR_2SFCA / f"g2sfca_score_{year}_{SCENARIO_TAG}_mw.csv"
        if fp_mw.exists():
            return fp_mw
        fp_orig = DIR_2SFCA / f"g2sfca_score_{year}_{SCENARIO_TAG}.csv"
        if fp_orig.exists():
            return fp_orig
            
    elif model_name == "Gravity":
        fp_mw = DIR_GRAVITY / f"gravity_score_{year}_{SCENARIO_TAG}_mw.csv"
        if fp_mw.exists():
            return fp_mw
        fp_orig = DIR_GRAVITY / f"gravity_score_{year}_{SCENARIO_TAG}.csv"
        if fp_orig.exists():
            return fp_orig

    return None

# 5. Spatial Weights Construction
- 서울시 14,979개 집계구 폴리곤 로드 및 광역 비교용 KNN(k=30) 공간가중행렬 구축

In [5]:
# 5. 공간 경계 로드 및 가중행렬 구축
gdf_seoul = load_seoul_boundary(BOUNDARY_FP)
print(f">> 서울시 집계구 경계 로드 완료: 총 {len(gdf_seoul):,}개 집계구")

w_knn30 = build_knn_weights(gdf_seoul, k=K_NEIGHBORS)
print(f">> 광역 패턴 비교용 KNN(k={K_NEIGHBORS}) 구축 완료 (Non-zero weights: {w_knn30.nonzero:,}개)")

>> 서울시 집계구 경계 로드 완료: 총 19,153개 집계구
>> 광역 패턴 비교용 KNN(k=30) 구축 완료 (Non-zero weights: 574,590개)


# 6. Batch Gi* Hotspot Pipeline (2 Models x 4 Years = 8 Runs)
- 2SFCA 및 Gravity 모형의 연도별 접근성 점수 로드 및 Gi* 연산
- 고속 벡터화 데이터프레임 생성 및 결과 파일(`_mw.csv`) 저장

In [6]:
# 6. 2개 모형 x 4개년 배치 실행
print("=" * 85)
print("RUNNING: TWO-MODEL (2SFCA vs GRAVITY) GI* HOTSPOT COMPARISON PIPELINE")
print("=" * 85)

model_targets = ["2SFCA", "Gravity"]
records = []

for model_name in model_targets:
    for year in YEARS:
        fp_score = resolve_model_score_file(model_name, year)
        if fp_score is None:
            print(f"  [!] 파일 누락 스킵: {model_name} | {year}")
            continue
            
        df_acc = pd.read_csv(fp_score, dtype={"oa_code": str})
        df_acc_reindexed = df_acc.set_index("oa_code").reindex(gdf_seoul["TOT_REG_CD"]).reset_index()
        y_scores = df_acc_reindexed["accessibility_score"].fillna(0.0).values
        
        # Gi* 분석 실행
        coded = compute_gi_star(y_scores, w_knn30, p_th=P_THRESHOLD)
        
        n_hot = (coded == "Hot Spot").sum()
        n_cold = (coded == "Cold Spot").sum()
        n_notsig = (coded == "Not Sig").sum()
        print(f"  [>] [{model_name:<7}] {year} | Hot: {n_hot:5,d} | Cold: {n_cold:5,d} | Not Sig: {n_notsig:5,d} <- {fp_score.name}")
        
        # 고속 벡터화 레코드 조립
        df_combo = pd.DataFrame({
            "model": model_name,
            "year": year,
            "oa_code": gdf_seoul["TOT_REG_CD"].values,
            "gi_class": coded
        })
        records.append(df_combo)

# 전체 병합 및 CSV 저장 (_mw)
df_two_model = pd.concat(records, ignore_index=True)
out_fp_mw = DIR_OUTPUT / "two_model_hotspot_k30_mw.csv"
df_two_model.to_csv(out_fp_mw, index=False, encoding="utf-8-sig")

print(f"\n>> 저장 완료: {out_fp_mw} (총 {len(df_two_model):,}행 = 8개 조합 x {len(gdf_seoul):,}개 집계구)")

RUNNING: TWO-MODEL (2SFCA vs GRAVITY) GI* HOTSPOT COMPARISON PIPELINE
  [>] [2SFCA  ] 2021 | Hot: 6,388 | Cold: 7,481 | Not Sig: 5,284 <- g2sfca_score_2021_week_낮_normal_mw.csv
  [>] [2SFCA  ] 2022 | Hot: 5,382 | Cold: 7,628 | Not Sig: 6,143 <- g2sfca_score_2022_week_낮_normal_mw.csv
  [>] [2SFCA  ] 2023 | Hot: 5,957 | Cold: 8,322 | Not Sig: 4,874 <- g2sfca_score_2023_week_낮_normal_mw.csv
  [>] [2SFCA  ] 2024 | Hot: 5,728 | Cold: 8,863 | Not Sig: 4,562 <- g2sfca_score_2024_week_낮_normal_mw.csv
  [>] [Gravity] 2021 | Hot: 5,758 | Cold: 7,607 | Not Sig: 5,788 <- gravity_score_2021_week_낮_normal_mw.csv
  [>] [Gravity] 2022 | Hot: 6,621 | Cold: 7,641 | Not Sig: 4,891 <- gravity_score_2022_week_낮_normal_mw.csv
  [>] [Gravity] 2023 | Hot: 6,390 | Cold: 7,798 | Not Sig: 4,965 <- gravity_score_2023_week_낮_normal_mw.csv
  [>] [Gravity] 2024 | Hot: 6,896 | Cold: 7,750 | Not Sig: 4,507 <- gravity_score_2024_week_낮_normal_mw.csv

>> 저장 완료: /mnt/cowork/EV/output/two_model_hotspot_k30_mw.csv (총 153,2

# 7. Summary Pivot Table (Model Pattern Comparison)
- 모형별·연도별 Hot Spot / Cold Spot 집계구 분포 빈도 및 비율(%) 피벗 요약표 출력

In [7]:
# 7. 모형별 군집 유형 분포 피벗 테이블
piv_count = df_two_model.pivot_table(
    index=["model", "year"],
    columns="gi_class",
    values="oa_code",
    aggfunc="count",
    fill_value=0
)[["Hot Spot", "Cold Spot", "Not Sig"]]

piv_pct = piv_count.div(len(gdf_seoul), axis=0) * 100

print("=" * 80)
print("             모형별·연도별 Gi* 군집 분포 집계표 (건수 / 비율 %)")
print("=" * 80)
summary_table = pd.concat([piv_count, piv_pct.add_suffix(" (%)")], axis=1)
display(summary_table)

             모형별·연도별 Gi* 군집 분포 집계표 (건수 / 비율 %)


gi_class      Hot Spot  Cold Spot  Not Sig  Hot Spot (%)  Cold Spot (%)  Not Sig (%)
model   year                                                                        
2SFCA   2021      6388       7481     5284       33.3525        39.0592      27.5884
        2022      5382       7628     6143       28.1000        39.8267      32.0733
        2023      5957       8322     4874       31.1022        43.4501      25.4477
        2024      5728       8863     4562       29.9065        46.2747      23.8187
Gravity 2021      5758       7607     5788       30.0632        39.7170      30.2198
        2022      6621       7641     4891       34.5690        39.8945      25.5365
        2023      6390       7798     4965       33.3629        40.7142      25.9228
        2024      6896       7750     4507       36.0048        40.4636      23.5316

# 8. Result Validation (Comparison with Original Outputs)
- 원본 산출물(`two_model_hotspot_k30.csv`)과 신규 산출물(`_mw.csv`) 간 8개 조합 핫스팟 분류 결과 일치율 전수 검증

In [ ]:
# 8. 원본 산출물 vs 신규 산출물(_mw) 정밀 일치율 검증
fp_orig = DIR_OUTPUT / "two_model_hotspot_k30.csv"
fp_new = DIR_OUTPUT / "two_model_hotspot_k30_mw.csv"

if not fp_orig.exists():
    print(f"[!] 비교할 원본 결과 파일이 존재하지 않습니다: {fp_orig}")
elif not fp_new.exists():
    print(f"[!] 신규 산출물 파일이 생성되지 않았습니다: {fp_new}")
else:
    df_orig = pd.read_csv(fp_orig, dtype={"oa_code": str})
    df_new = pd.read_csv(fp_new, dtype={"oa_code": str})
    
    comp = df_orig.merge(df_new, on=["model", "year", "oa_code"], suffixes=("_orig", "_new"))
    comp["is_match"] = comp["gi_class_orig"] == comp["gi_class_new"]
    
    total_records = len(comp)
    match_count = comp["is_match"].sum()
    overall_match_rate = (match_count / total_records) * 100
    
    print("=" * 80)
    print("           2개 모형 Gi* 비교 산출물 원본 vs 리팩토링 코드 수치 검증 요약표")
    print("=" * 80)
    print(f"- 총 검증 레코드 수  : {total_records:,}개 (8개 조합 x {len(gdf_seoul):,}개)")
    print(f"- 군집 분류 일치 건수: {match_count:,}개")
    print(f"- 전체 일치율 (Match Rate): {overall_match_rate:.4f}%")
    
    # 세부 조합별 일치율
    piv_val = comp.groupby(["model", "year"])["is_match"].agg(
        총레코드="count", 
        일치건수="sum", 
        일치율=lambda x: f"{(x.mean()*100):.2f}%"
    )
    display(piv_val)
    
    if match_count == total_records:
        print(">> [판정] 2SFCA 및 Gravity 모형의 모든 공간 핫스팟/콜드스팟 분류가 원본 산출물과 100% 완벽히 일치합니다.")
    else:
        print(">> [판정] 일부 분류 차이가 발생했습니다. 불일치 샘플을 점검해주세요.")
        display(comp[~comp["is_match"]].head(5))

           2개 모형 Gi* 비교 산출물 원본 vs 리팩토링 코드 수치 검증 요약표
- 총 검증 레코드 수  : 153,224개 (8개 조합 x 19,153개)
- 군집 분류 일치 건수: 153,224개
- 전체 일치율 (Match Rate): 100.0000%


총레코드   일치건수      일치율
model   year                       
2SFCA   2021  19153  19153  100.00%
        2022  19153  19153  100.00%
        2023  19153  19153  100.00%
        2024  19153  19153  100.00%
Gravity 2021  19153  19153  100.00%
        2022  19153  19153  100.00%
        2023  19153  19153  100.00%
        2024  19153  19153  100.00%

>> [판정] 2SFCA 및 Gravity 모형의 모든 공간 핫스팟/콜드스팟 분류가 원본 산출물과 100% 완벽히 일치합니다.
